In [36]:
# ------------------------------
# === Importy a základní funkce ===
# ------------------------------
import os, re
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

def _read_text_guess_encoding(path: str) -> str:
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-8", "cp1250", "latin-1"):
        try:
            return raw.decode(enc, errors="ignore")
        except Exception:
            continue
    return raw.decode("latin-1", errors="ignore")

def _extract_xml(text: str) -> str:
    start = text.find("<?xml")
    if start == -1:
        raise ValueError("Soubor neobsahuje XML sekci.")
    return text[start:]

def _to_number(x):
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).replace(",", ".").strip()
    try:
        return float(s)
    except:
        return np.nan

def parse_drp_to_dataframe(path: str) -> pd.DataFrame:
    text = _read_text_guess_encoding(path)
    xml_str = _extract_xml(text)
    root = ET.fromstring(xml_str.replace("\x00", ""))

    data_labels = {
        child.tag: child.text.strip()
        for child in root if re.match(r"^Data\d+$", child.tag) and (child.text or "").strip()
    }

    records = [
        {elem.tag: (elem.text or "").strip() for elem in m}
        for m in root.findall(".//MessObj")
    ]

    if not records:
        raise ValueError("Nenalezeny položky <MessObj>.")

    df = pd.DataFrame(records).rename(columns=data_labels)
    for col in df.columns:
        df[col] = df[col].map(_to_number)

    return df.reset_index(drop=True)

def ensure_monotonic(df, pos_col):
    return df.sort_values(pos_col).reset_index(drop=True)

def normalize_pos(df, col, unit="km"):
    df = df.copy()
    df["Pos_m"] = df[col] * (1000 if unit == "km" else 1)
    return df

def interpolate_geo_to_grid(df_geo, pos_col="Pos_m", step=0.25):
    g = np.arange(df_geo[pos_col].min(), df_geo[pos_col].max() + step, step)
    out = pd.DataFrame({pos_col: g})
    num_cols = [c for c in df_geo.columns if c != pos_col and pd.api.types.is_numeric_dtype(df_geo[c])]
    for c in num_cols:
        out[c] = np.interp(g, df_geo[pos_col], df_geo[c], left=np.nan, right=np.nan)
    return out

def agg_drp_to_geo(df_drp, df_geo, src="Pos", tgt="Pos_m"):
    df_drp = ensure_monotonic(df_drp, src)
    df_geo = ensure_monotonic(df_geo, tgt)
    t = df_geo.copy()
    t["LeftBound"] = (df_geo[tgt].shift(1) + df_geo[tgt]) / 2
    t["RightBound"] = (df_geo[tgt] + df_geo[tgt].shift(-1)) / 2
    t.iloc[0, t.columns.get_loc("LeftBound")] = df_drp[src].min()
    t.iloc[-1, t.columns.get_loc("RightBound")] = df_drp[src].max()

    numeric_cols = [
        c for c in df_drp.columns
        if c not in [src] and pd.api.types.is_numeric_dtype(df_drp[c])
    ]

    rows = []
    for _, r in t.iterrows():
        seg = df_drp[(df_drp[src] >= r["LeftBound"]) & (df_drp[src] < r["RightBound"])]
        out = {tgt: r[tgt]}
        for c in numeric_cols:
            vals = seg[c].dropna()
            out[c] = vals.median() if len(vals) else np.nan
        rows.append(out)

    return pd.DataFrame(rows)

def detect_direction(df, pos_col="Pos"):
    return 1 if df[pos_col].iloc[-1] > df[pos_col].iloc[0] else -1

def adjust_control_band(control_band, direction):
    if pd.isna(control_band):
        return np.nan
    return control_band if direction == 1 else ("L" if control_band == "P" else "P")


In [ ]:

# ------------------------------
# === Nahrát GEO + DRP (sekvenčně) ===
# ------------------------------
from google.colab import files

print("Nahraj GEO 1 (.xlsx)…")
geo1_path = next(iter(files.upload().keys()))

print("Nahraj GEO 2 (.xlsx)…")
geo2_path = next(iter(files.upload().keys()))

print("Nahraj GEO 3 (.xlsx)…")
geo3_path = next(iter(files.upload().keys()))

print("Nahraj GEO 4 - Kontrolní(.xlsx)…")
geo4_path = next(iter(files.upload().keys()))

print("Nahraj DRP 1 (.drp)…")
drp1_path = next(iter(files.upload().keys()))

print("Nahraj DRP 2 (.drp)…")
drp2_path = next(iter(files.upload().keys()))


In [ ]:
# ------------------------------
# === Zpracování všech částí ===
# ------------------------------
workflow_mode      = "geo_to_0p25m"
geodet_pos_col     = "stan [km]"
geodet_pos_unit    = "km"
grid_step_m        = 0.25

# === Pomocné funkce ===
def compute_alignment_offset(df_geo, pos_col="Pos_m", step=0.25):
    real_start = df_geo[pos_col].iloc[0]
    aligned_start = round(real_start / step) * step
    return aligned_start - real_start

def apply_offset(df, offset, pos_col):
    df = df.copy()
    df[pos_col] = df[pos_col] + offset
    return df

def load_geodet(path):
    df_raw = pd.read_excel(path)
    df_raw.columns = [str(c).strip() for c in df_raw.columns]
    col = geodet_pos_col if geodet_pos_col in df_raw.columns else df_raw.columns[0]
    df = normalize_pos(df_raw, col, geodet_pos_unit)
    return df_raw, ensure_monotonic(df, "Pos_m")

def fill_numeric_empty_with_zero(df):
    df = df.copy()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(0)
    return df

def build_interpolated_geo(df_geo, geo_tag):
    geo_keep = [
        "Pos_m",
        "stan [km]",
        "posun [mm]",
        "zdvih [mm]",
        "L_pás [mm]",
        "P_pás [mm]",
    ]
    geo_keep_existing = [c for c in geo_keep if c in df_geo.columns]

    if "Pos_m" not in geo_keep_existing:
        raise KeyError(f"V GEO datech {geo_tag} chybí sloupec 'Pos_m'.")

    df_geo_final = interpolate_geo_to_grid(
        df_geo[geo_keep_existing].copy(),
        step=grid_step_m
    )

    df_geo_final = df_geo_final[
        (df_geo_final["Pos_m"] >= common_start) & (df_geo_final["Pos_m"] <= common_end)
    ].copy()

    df_geo_final = df_geo_final.rename(
        columns={c: f"{c}{geo_tag}" for c in df_geo_final.columns if c != "Pos_m"}
    )

    return df_geo_final

# === 1. Načtení dat ===
df_geo1_raw, df_geo1 = load_geodet(geo1_path)
df_geo2_raw, df_geo2 = load_geodet(geo2_path)
df_geo3_raw, df_geo3 = load_geodet(geo3_path)
df_geo4_raw, df_geo4 = load_geodet(geo4_path)

df_drp1 = parse_drp_to_dataframe(drp1_path)
df_drp2 = parse_drp_to_dataframe(drp2_path)

# === 2. Offsety ===
offset1 = compute_alignment_offset(df_geo1, "Pos_m", grid_step_m)
offset2 = compute_alignment_offset(df_geo2, "Pos_m", grid_step_m)
offset3 = compute_alignment_offset(df_geo3, "Pos_m", grid_step_m)
offset4 = compute_alignment_offset(df_geo4, "Pos_m", grid_step_m)

df_geo1 = apply_offset(df_geo1, offset1, "Pos_m")
df_geo2 = apply_offset(df_geo2, offset2, "Pos_m")
df_geo3 = apply_offset(df_geo3, offset3, "Pos_m")
df_geo4 = apply_offset(df_geo4, offset4, "Pos_m")

df_drp1 = ensure_monotonic(df_drp1, "Pos")
df_drp2 = ensure_monotonic(df_drp2, "Pos")

df_drp1["Pos"] = df_drp1["Pos"] + offset1
df_drp2["Pos"] = df_drp2["Pos"] + offset2

# === 3. Výpočet průniku ===
common_start = max(
    df_geo1["Pos_m"].min(), df_geo2["Pos_m"].min(),
    df_geo3["Pos_m"].min(), df_geo4["Pos_m"].min(),
    df_drp1["Pos"].min(), df_drp2["Pos"].min()
)
common_end = min(
    df_geo1["Pos_m"].max(), df_geo2["Pos_m"].max(),
    df_geo3["Pos_m"].max(), df_geo4["Pos_m"].max(),
    df_drp1["Pos"].max(), df_drp2["Pos"].max()
)

# === 4. Funkce merge_pair bez Řp, bez rampové logiky ===
def merge_pair(df_geo, df_drp, geo_tag, drp_tag):
    geo_keep = [
        "Pos_m",
        "stan [km]",
        "posun [mm]",
        "zdvih [mm]",
        "L_pás [mm]",
        "P_pás [mm]",
    ]
    geo_keep_existing = [c for c in geo_keep if c in df_geo.columns]

    if "Pos_m" not in geo_keep_existing:
        raise KeyError("V GEO datech chybí sloupec 'Pos_m'.")

    grid = interpolate_geo_to_grid(df_geo[geo_keep_existing].copy(), step=grid_step_m)

    merged = pd.merge_asof(
        grid.sort_values("Pos_m"),
        df_drp.sort_values("Pos"),
        left_on="Pos_m",
        right_on="Pos",
        direction="nearest"
    )

    merged = merged[
        (merged["Pos_m"] >= common_start) & (merged["Pos_m"] <= common_end)
    ].copy()

    final_cols = {"Pos_m": "Pos_m"}
    for col in merged.columns:
        if col in ["Pos_m", "Pos"]:
            continue
        if col in geo_keep_existing:
            final_cols[col] = f"{col}{geo_tag}"
        else:
            final_cols[col] = f"{col}{drp_tag}" if not col.endswith(drp_tag) else col

    return merged.rename(columns=final_cols)

# === 5. Finální tabulky ===
merged_G1_D1 = merge_pair(df_geo1, df_drp1, "_G1", "_D1")
merged_G2_D2 = merge_pair(df_geo2, df_drp2, "_G2", "_D2")

df_geo3_final = build_interpolated_geo(df_geo3, "_G3")
df_geo4_final = build_interpolated_geo(df_geo4, "_G4")

# === 6. Vyplnění prázdných buněk nulou ===
merged_G1_D1 = fill_numeric_empty_with_zero(merged_G1_D1)
merged_G2_D2 = fill_numeric_empty_with_zero(merged_G2_D2)
df_geo3_final = fill_numeric_empty_with_zero(df_geo3_final)
df_geo4_final = fill_numeric_empty_with_zero(df_geo4_final)

print(f"Hotovo. Rozsah dat: {common_start} - {common_end}")

In [ ]:
# ------------------------------
# === Uložit výsledky ===
# ------------------------------
def round_numeric_cols(df, decimals=2):
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].round(decimals)
    return df

out_name = "Merched_Analyza_K1.xlsx"
with pd.ExcelWriter(out_name) as writer:

    round_numeric_cols(merged_G1_D1).to_excel(writer, sheet_name="G1_vs_D1", index=False)
    round_numeric_cols(merged_G2_D2).to_excel(writer, sheet_name="G2_vs_D2", index=False)

    round_numeric_cols(df_geo3_final).to_excel(writer, sheet_name="G3_final", index=False)
    round_numeric_cols(df_geo4_final).to_excel(writer, sheet_name="G4_control", index=False)

print(f"Uloženo: {out_name}")
